In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import train_test_split, StratifiedKFold
import os

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

Device: cuda
GPU: Tesla T4


In [2]:
train_df = pd.read_csv('/kaggle/input/datasets/vladcvx/sentiment-classification-data-256/data1/train.csv')
test_df  = pd.read_csv('/kaggle/input/datasets/vladcvx/sentiment-classification-data-256/data1/test.csv')

import re
def clean_text(text):
    return re.sub(r'\s+', ' ', text).strip()
train_df['sentence'] = train_df['sentence'].apply(clean_text)
test_df['sentence']  = test_df['sentence'].apply(clean_text)

_, _, _, val_labels = train_test_split(
    train_df['sentence'].tolist(),
    train_df['label'].tolist(),
    test_size=0.10,
    random_state=SEED,
    stratify=train_df['label'].tolist(),
)
y_val = np.array(val_labels)
print(f'Train: {len(train_df):,} | Test: {len(test_df):,} | Val: {len(y_val):,}')

Train: 252,000 | Test: 168,000 | Val: 25,200


In [3]:
SEP_DIR    = '/kaggle/input/datasets/vladcvx/cil-sep'
CONCAT_DIR = '/kaggle/input/datasets/vladcvx/bert-roberta-val-test-probs'

# ── SEP ───────────────────────────────────────────────────────────────────────
bert_sep_val          = np.load(f'{SEP_DIR}/bert_probs_val.npy')
roberta1_sep_val      = np.load(f'{SEP_DIR}/roberta_probs_val.npy')
roberta2_sep_val      = np.load(f'{SEP_DIR}/roberta_higher_lr_probs_val.npy')
electra_sep_val       = np.load(f'{SEP_DIR}/electra_probs_val.npy')
distilbert_sep_val    = np.load(f'{SEP_DIR}/distilbert_probs_val.npy')

bert_sep_test         = np.load(f'{SEP_DIR}/bert_probs_test.npy')
roberta1_sep_test     = np.load(f'{SEP_DIR}/roberta_probs_test.npy')
roberta2_sep_test     = np.load(f'{SEP_DIR}/roberta_higher_lr_probs_test.npy')
electra_sep_test      = np.load(f'{SEP_DIR}/electra_probs_test.npy')
distilbert_sep_test   = np.load(f'{SEP_DIR}/distilbert_probs_test.npy')

# ── Concat ────────────────────────────────────────────────────────────────────
bert_concat_val        = np.load(f'{CONCAT_DIR}/bert_probs_val.npy')
roberta2_concat_val    = np.load(f'{CONCAT_DIR}/roberta_2_epochs_higher_lr_probs_val.npy')
roberta5_concat_val    = np.load(f'{CONCAT_DIR}/roberta_5_epochs_probs_val.npy')
electra_concat_val     = np.load(f'{CONCAT_DIR}/electra_probs_val.npy')
distilbert_concat_val  = np.load(f'{CONCAT_DIR}/distilbert_probs_val.npy')

bert_concat_test       = np.load(f'{CONCAT_DIR}/bert_probs_test.npy')
roberta2_concat_test   = np.load(f'{CONCAT_DIR}/roberta_2_epochs_higher_lr_probs_test.npy')
roberta5_concat_test   = np.load(f'{CONCAT_DIR}/roberta_5_epochs_probs_test.npy')
electra_concat_test    = np.load(f'{CONCAT_DIR}/electra_probs_test.npy')
distilbert_concat_test = np.load(f'{CONCAT_DIR}/distilbert_probs_test.npy')

print('All probs loaded. Shapes:')
for name, arr in [
    ('bert_sep_val',           bert_sep_val),
    ('roberta1_sep_val',       roberta1_sep_val),
    ('roberta2_sep_val',       roberta2_sep_val),
    ('electra_sep_val',        electra_sep_val),
    ('distilbert_sep_val',     distilbert_sep_val),
    ('bert_concat_val',        bert_concat_val),
    ('roberta2_concat_val',    roberta2_concat_val),
    ('roberta5_concat_val',    roberta5_concat_val),
    ('electra_concat_val',     electra_concat_val),
    ('distilbert_concat_val',  distilbert_concat_val),
]:
    print(f'  {name:<30}: {arr.shape}')

All probs loaded. Shapes:
  bert_sep_val                  : (25200, 5)
  roberta1_sep_val              : (25200, 5)
  roberta2_sep_val              : (25200, 5)
  electra_sep_val               : (25200, 5)
  distilbert_sep_val            : (25200, 5)
  bert_concat_val               : (25200, 5)
  roberta2_concat_val           : (25200, 5)
  roberta5_concat_val           : (25200, 5)
  electra_concat_val            : (25200, 5)
  distilbert_concat_val         : (25200, 5)


In [4]:
X_full = np.hstack([
    bert_sep_val,  roberta1_sep_val,  roberta2_sep_val,  electra_sep_val,  distilbert_sep_val,
    bert_concat_val, roberta2_concat_val, roberta5_concat_val, electra_concat_val, distilbert_concat_val,
])  # (N_val, 50)

X_test = np.hstack([
    bert_sep_test,  roberta1_sep_test,  roberta2_sep_test,  electra_sep_test,  distilbert_sep_test,
    bert_concat_test, roberta2_concat_test, roberta5_concat_test, electra_concat_test, distilbert_concat_test,
])  # (N_test, 50)

X_test_t = torch.tensor(X_test, dtype=torch.float32).to(device)

print(f'X_full : {X_full.shape}')
print(f'X_test : {X_test.shape}')

X_full : (25200, 50)
X_test : (168000, 50)


In [5]:
def evaluation_score(preds, labels):
    mae = np.mean(np.abs(np.array(preds) - np.array(labels)))
    return 1.0 - mae / 4.0

def make_submission(preds, filename):
    df = pd.DataFrame({
        'id':    test_df['id'].astype(int),
        'label': preds.astype(int),
    })
    assert len(df) == len(test_df)
    assert df['label'].isin([0, 1, 2, 3, 4]).all()
    df.to_csv(f'/kaggle/working/{filename}', index=False)
    print(f'Saved: {filename}')
    return df

def train_mlp(model, X_tr_t, y_tr_t, X_ho_t, y_ho,
              epochs=300, lr=1e-3, weight_decay=1e-2):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.CrossEntropyLoss()
    best_score, best_state = 0.0, None
    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        criterion(model(X_tr_t), y_tr_t).backward()
        optimizer.step()
        model.eval()
        with torch.no_grad():
            preds_ho = torch.argmax(model(X_ho_t), dim=1).cpu().numpy()
        score = evaluation_score(preds_ho, y_ho)
        if score > best_score:
            best_score = score
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    return model, best_score

In [6]:
# ── Weighted average — weights optimised on val set ───────────────────────────
# Start with equal weights (0.1 each), tune manually or via scipy.optimize
# Order: bert_sep, roberta1_sep, roberta2_sep, electra_sep, distilbert_sep,
#        bert_concat, roberta2_concat, roberta5_concat, electra_concat, distilbert_concat
W = np.array([0.10, 0.15, 0.15, 0.10, 0.10,
              0.10, 0.10, 0.10, 0.05, 0.05])
W = W / W.sum()  # normalise to sum=1

all_val = np.stack([
    bert_sep_val,  roberta1_sep_val,  roberta2_sep_val,  electra_sep_val,  distilbert_sep_val,
    bert_concat_val, roberta2_concat_val, roberta5_concat_val, electra_concat_val, distilbert_concat_val,
], axis=0)  # (10, N_val, 5)

all_test = np.stack([
    bert_sep_test,  roberta1_sep_test,  roberta2_sep_test,  electra_sep_test,  distilbert_sep_test,
    bert_concat_test, roberta2_concat_test, roberta5_concat_test, electra_concat_test, distilbert_concat_test,
], axis=0)  # (10, N_test, 5)

# Weighted average: (10, N, 5) * (10, 1, 1) -> sum over axis 0
weighted_val_probs  = (all_val  * W[:, None, None]).sum(axis=0)  # (N_val,  5)
weighted_test_probs = (all_test * W[:, None, None]).sum(axis=0)  # (N_test, 5)

weighted_val_preds = weighted_val_probs.argmax(axis=1)
score_weighted_avg = evaluation_score(weighted_val_preds, y_val)
print(f'Weighted avg val score: {score_weighted_avg:.4f}  (weights: {W.round(3)})')

preds_weighted = weighted_test_probs.argmax(axis=1)
make_submission(preds_weighted, 'submission_weighted_avg_10models.csv')

Weighted avg val score: 0.9099  (weights: [0.1  0.15 0.15 0.1  0.1  0.1  0.1  0.1  0.05 0.05])
Saved: submission_weighted_avg_10models.csv


,id,label
0,0,1
1,1,1
2,2,3
3,3,1
4,4,1
...,...,...
167995,167995,1
167996,167996,4
167997,167997,1
167998,167998,4


In [7]:
# Holdout split (reused across honest variants)
idx_tr, idx_ho = train_test_split(
    np.arange(len(y_val)), test_size=0.3, random_state=42, stratify=y_val
)
X_tr, X_ho = X_full[idx_tr], X_full[idx_ho]
y_tr, y_ho = y_val[idx_tr],  y_val[idx_ho]

X_tr_t = torch.tensor(X_tr, dtype=torch.float32).to(device)
y_tr_t = torch.tensor(y_tr, dtype=torch.long).to(device)
X_ho_t = torch.tensor(X_ho, dtype=torch.float32).to(device)

print('Holdout split ready.')
print(f'  Train: {len(idx_tr):,} | Holdout: {len(idx_ho):,}')

Holdout split ready.
  Train: 17,640 | Holdout: 7,560


In [8]:
class SmallMLP(nn.Module):
    def __init__(self, input_dim=50, hidden_dim=1024, num_classes=5):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, num_classes),
        )
    def forward(self, x): return self.mlp(x)

mlp_honest = SmallMLP(input_dim=50, hidden_dim=1024).to(device)
mlp_honest, score_honest = train_mlp(mlp_honest, X_tr_t, y_tr_t, X_ho_t, y_ho)
print(f'mlp_honest holdout score: {score_honest:.4f}')

mlp_honest.eval()
with torch.no_grad():
    preds = torch.argmax(mlp_honest(X_test_t), dim=1).cpu().numpy()
make_submission(preds, 'submission_mlp_honest_10models.csv')

mlp_honest holdout score: 0.9098
Saved: submission_mlp_honest_10models.csv


,id,label
0,0,1
1,1,1
2,2,3
3,3,1
4,4,1
...,...,...
167995,167995,1
167996,167996,4
167997,167997,1
167998,167998,4


In [9]:
class BigMLP(nn.Module):
    def __init__(self, input_dim=50, hidden_dim=4096, num_classes=5):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim // 2, hidden_dim // 4),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim // 4, num_classes),
        )
    def forward(self, x): return self.mlp(x)

big_mlp = BigMLP(input_dim=50, hidden_dim=4096).to(device)
big_mlp, score_big = train_mlp(big_mlp, X_tr_t, y_tr_t, X_ho_t, y_ho,
                                epochs=300, lr=1e-3)
print(f'honest_big_mlp holdout score: {score_big:.4f}')

big_mlp.eval()
with torch.no_grad():
    preds = torch.argmax(big_mlp(X_test_t), dim=1).cpu().numpy()
make_submission(preds, 'submission_honest_big_mlp_10models.csv')

honest_big_mlp holdout score: 0.9107
Saved: submission_honest_big_mlp_10models.csv


,id,label
0,0,1
1,1,1
2,2,3
3,3,1
4,4,1
...,...,...
167995,167995,1
167996,167996,4
167997,167997,1
167998,167998,4


In [10]:
class MediumMLP(nn.Module):
    def __init__(self, input_dim=50, hidden_dim=2048, num_classes=5):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim // 2, hidden_dim // 4),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim // 4, num_classes),
        )
    def forward(self, x): return self.mlp(x)

med_mlp = MediumMLP(input_dim=50, hidden_dim=2048).to(device)
med_mlp, score_med = train_mlp(med_mlp, X_tr_t, y_tr_t, X_ho_t, y_ho,
                                epochs=300, lr=1e-3)
print(f'mlp_honest_10models holdout score: {score_med:.4f}')

med_mlp.eval()
with torch.no_grad():
    preds = torch.argmax(med_mlp(X_test_t), dim=1).cpu().numpy()
make_submission(preds, 'submission_mlp_honest_10models_medium.csv')

mlp_honest_10models holdout score: 0.9105
Saved: submission_mlp_honest_10models_medium.csv


,id,label
0,0,1
1,1,1
2,2,3
3,3,1
4,4,1
...,...,...
167995,167995,1
167996,167996,4
167997,167997,1
167998,167998,4


In [11]:
N_FOLDS_4 = 5
kf4 = StratifiedKFold(n_splits=N_FOLDS_4, shuffle=True, random_state=42)

oof_preds_kf4  = np.zeros(len(y_val))
test_probs_kf4 = []
fold_scores_kf4 = []

for fold, (idx_tr, idx_ho) in enumerate(kf4.split(X_full, y_val)):
    X_tr_f, X_ho_f = X_full[idx_tr], X_full[idx_ho]
    y_tr_f, y_ho_f = y_val[idx_tr],  y_val[idx_ho]

    X_tr_ft = torch.tensor(X_tr_f, dtype=torch.float32).to(device)
    y_tr_ft = torch.tensor(y_tr_f, dtype=torch.long).to(device)
    X_ho_ft = torch.tensor(X_ho_f, dtype=torch.float32).to(device)

    mlp_f = MediumMLP(input_dim=50, hidden_dim=2048).to(device)
    mlp_f, best_score_f = train_mlp(mlp_f, X_tr_ft, y_tr_ft, X_ho_ft, y_ho_f,
                                     epochs=300, lr=1e-3)
    fold_scores_kf4.append(best_score_f)

    mlp_f.eval()
    with torch.no_grad():
        oof_preds_kf4[idx_ho] = torch.argmax(mlp_f(X_ho_ft), dim=1).cpu().numpy()
        test_probs_kf4.append(torch.softmax(mlp_f(X_test_t), dim=1).cpu().numpy())

    print(f'  Fold {fold+1} score: {best_score_f:.4f}')

oof_score_kf4 = evaluation_score(oof_preds_kf4, y_val)
print(f'\nOOF score: {oof_score_kf4:.4f}  |  mean fold: {np.mean(fold_scores_kf4):.4f} ± {np.std(fold_scores_kf4):.4f}')

preds_kf4 = np.argmax(np.mean(test_probs_kf4, axis=0), axis=1)
make_submission(preds_kf4, 'submission_mlp_kfold_10models.csv')

  Fold 1 score: 0.9122
  Fold 2 score: 0.9121
  Fold 3 score: 0.9116
  Fold 4 score: 0.9079
  Fold 5 score: 0.9122

OOF score: 0.9112  |  mean fold: 0.9112 ± 0.0016
Saved: submission_mlp_kfold_10models.csv


,id,label
0,0,1
1,1,1
2,2,3
3,3,1
4,4,1
...,...,...
167995,167995,1
167996,167996,4
167997,167997,1
167998,167998,4


In [12]:
N_FOLDS_5 = 10
EPOCHS_5  = 500
SEEDS_5   = [42, 123, 7]

all_test_probs_5 = []
all_oof_preds_5  = np.zeros((len(SEEDS_5), len(y_val)))

for s_idx, seed in enumerate(SEEDS_5):
    print(f'\n===== Seed {seed} =====')
    torch.manual_seed(seed)
    np.random.seed(seed)

    kf5 = StratifiedKFold(n_splits=N_FOLDS_5, shuffle=True, random_state=seed)
    fold_scores_5 = []

    for fold, (idx_tr, idx_ho) in enumerate(kf5.split(X_full, y_val)):
        X_tr_f, X_ho_f = X_full[idx_tr], X_full[idx_ho]
        y_tr_f, y_ho_f = y_val[idx_tr],  y_val[idx_ho]

        X_tr_ft = torch.tensor(X_tr_f, dtype=torch.float32).to(device)
        y_tr_ft = torch.tensor(y_tr_f, dtype=torch.long).to(device)
        X_ho_ft = torch.tensor(X_ho_f, dtype=torch.float32).to(device)

        mlp_f = MediumMLP(input_dim=50, hidden_dim=2048).to(device)
        mlp_f, best_score_f = train_mlp(mlp_f, X_tr_ft, y_tr_ft, X_ho_ft, y_ho_f,
                                         epochs=EPOCHS_5, lr=1e-3)
        fold_scores_5.append(best_score_f)

        mlp_f.eval()
        with torch.no_grad():
            all_oof_preds_5[s_idx][idx_ho] = torch.argmax(mlp_f(X_ho_ft), dim=1).cpu().numpy()
            all_test_probs_5.append(torch.softmax(mlp_f(X_test_t), dim=1).cpu().numpy())

        print(f'  Fold {fold+1:2d} score: {best_score_f:.4f}')

    oof_score_5 = evaluation_score(all_oof_preds_5[s_idx], y_val)
    print(f'Seed {seed} OOF: {oof_score_5:.4f}  |  mean fold: {np.mean(fold_scores_5):.4f} ± {np.std(fold_scores_5):.4f}')

mean_oof_preds_5 = all_oof_preds_5.mean(axis=0).round().astype(int).clip(0, 4)
final_oof_score_5 = evaluation_score(mean_oof_preds_5, y_val)
print(f'\nFinal OOF (avg across {len(SEEDS_5)} seeds): {final_oof_score_5:.4f}')

preds_kfold_seeds = np.argmax(np.mean(all_test_probs_5, axis=0), axis=1)
make_submission(preds_kfold_seeds, 'submission_mlp_kfold_seeds_10models.csv')


===== Seed 42 =====
  Fold  1 score: 0.9121
  Fold  2 score: 0.9127
  Fold  3 score: 0.9138
  Fold  4 score: 0.9102
  Fold  5 score: 0.9108
  Fold  6 score: 0.9132
  Fold  7 score: 0.9053
  Fold  8 score: 0.9120
  Fold  9 score: 0.9111
  Fold 10 score: 0.9140
Seed 42 OOF: 0.9115  |  mean fold: 0.9115 ± 0.0024

===== Seed 123 =====
  Fold  1 score: 0.9101
  Fold  2 score: 0.9127
  Fold  3 score: 0.9179
  Fold  4 score: 0.9103
  Fold  5 score: 0.9107
  Fold  6 score: 0.9110
  Fold  7 score: 0.9099
  Fold  8 score: 0.9126
  Fold  9 score: 0.9095
  Fold 10 score: 0.9111
Seed 123 OOF: 0.9116  |  mean fold: 0.9116 ± 0.0023

===== Seed 7 =====
  Fold  1 score: 0.9139
  Fold  2 score: 0.9086
  Fold  3 score: 0.9105
  Fold  4 score: 0.9145
  Fold  5 score: 0.9110
  Fold  6 score: 0.9126
  Fold  7 score: 0.9095
  Fold  8 score: 0.9113
  Fold  9 score: 0.9132
  Fold 10 score: 0.9103
Seed 7 OOF: 0.9115  |  mean fold: 0.9115 ± 0.0018

Final OOF (avg across 3 seeds): 0.9119
Saved: submission_mlp_kf

,id,label
0,0,1
1,1,1
2,2,3
3,3,1
4,4,1
...,...,...
167995,167995,1
167996,167996,4
167997,167997,1
167998,167998,4


In [13]:
print('=' * 55)
print('SUBMISSION SUMMARY (val/OOF scores)')
print('=' * 55)
print(f'0. weighted_avg        (tunable weights)  : {score_weighted_avg:.4f}')
print(f'1. mlp_honest          (small,  holdout)  : {score_honest:.4f}')
print(f'2. honest_big_mlp      (big,    holdout)  : {score_big:.4f}')
print(f'3. mlp_honest_10models (medium, holdout)  : {score_med:.4f}')
print(f'4. kfold               (5-fold, seed=42)  : {oof_score_kf4:.4f}')
print(f'5. kfold_seeds         (10-fold, 3 seeds) : {final_oof_score_5:.4f}')

SUBMISSION SUMMARY (val/OOF scores)
0. weighted_avg        (tunable weights)  : 0.9099
1. mlp_honest          (small,  holdout)  : 0.9098
2. honest_big_mlp      (big,    holdout)  : 0.9107
3. mlp_honest_10models (medium, holdout)  : 0.9105
4. kfold               (5-fold, seed=42)  : 0.9112
5. kfold_seeds         (10-fold, 3 seeds) : 0.9119
